In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Read files
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')

# --- metric.csv summary ---
metric_row_count = len(df_metric)

# distinct cmdb_id values (sorted)
metric_cmdb_ids = sorted(df_metric['cmdb_id'].dropna().unique().tolist())

# kpi_name counts (top 50)
metric_kpi_counts = df_metric['kpi_name'].value_counts().reset_index()
metric_kpi_counts.columns = ['kpi_name', 'count']
metric_kpi_counts = metric_kpi_counts.head(50)

# min/max timestamp and convert to Asia/Shanghai (UTC+8)
if not df_metric['timestamp'].empty:
    metric_min_ts = int(df_metric['timestamp'].min())
    metric_max_ts = int(df_metric['timestamp'].max())
    metric_min_ts_local = pd.to_datetime(metric_min_ts, unit='s', utc=True).tz_convert(tz)
    metric_max_ts_local = pd.to_datetime(metric_max_ts, unit='s', utc=True).tz_convert(tz)
else:
    metric_min_ts = metric_max_ts = None
    metric_min_ts_local = metric_max_ts_local = None

# up to 5 example rows (compact): include original timestamp and local time
metric_examples = df_metric.head(5).copy()
if not metric_examples.empty:
    metric_examples['local_time'] = pd.to_datetime(metric_examples['timestamp'], unit='s', utc=True).dt.tz_convert(tz).astype(str)
    metric_examples = metric_examples[['timestamp', 'local_time', 'cmdb_id', 'kpi_name', 'value']]

# --- trace.csv summary ---
trace_row_count = len(df_trace)

# distinct cmdb_id values (sorted)
trace_cmdb_ids = sorted(df_trace['cmdb_id'].dropna().unique().tolist())

# trace_name counts (top 50)
trace_name_counts = df_trace['trace_name'].value_counts().reset_index()
trace_name_counts.columns = ['trace_name', 'count']
trace_name_counts = trace_name_counts.head(50)

# min/max timestamp and convert to Asia/Shanghai (UTC+8)
if not df_trace['timestamp'].empty:
    trace_min_ts = int(df_trace['timestamp'].min())
    trace_max_ts = int(df_trace['timestamp'].max())
    trace_min_ts_local = pd.to_datetime(trace_min_ts, unit='s', utc=True).tz_convert(tz)
    trace_max_ts_local = pd.to_datetime(trace_max_ts, unit='s', utc=True).tz_convert(tz)
else:
    trace_min_ts = trace_max_ts = None
    trace_min_ts_local = trace_max_ts_local = None

# up to 5 example rows (compact): include original timestamp and local time
trace_examples = df_trace.head(5).copy()
if not trace_examples.empty:
    trace_examples['local_time'] = pd.to_datetime(trace_examples['timestamp'], unit='s', utc=True).dt.tz_convert(tz).astype(str)
    trace_examples = trace_examples[['timestamp', 'local_time', 'cmdb_id', 'trace_name', 'value']]

# Assemble compact summary
summary = {
    'metric': {
        'row_count': metric_row_count,
        'distinct_cmdb_ids': metric_cmdb_ids,
        'kpi_name_counts_top': metric_kpi_counts,
        'min_timestamp_epoch': metric_min_ts,
        'max_timestamp_epoch': metric_max_ts,
        'min_timestamp_local': metric_min_ts_local,
        'max_timestamp_local': metric_max_ts_local,
        'example_rows': metric_examples
    },
    'trace': {
        'row_count': trace_row_count,
        'distinct_cmdb_ids': trace_cmdb_ids,
        'trace_name_counts_top': trace_name_counts,
        'min_timestamp_epoch': trace_min_ts,
        'max_timestamp_epoch': trace_max_ts,
        'min_timestamp_local': trace_min_ts_local,
        'max_timestamp_local': trace_max_ts_local,
        'example_rows': trace_examples
    }
}

# Display the compact summary variable (IPython will render DataFrames concisely)
summary

```
Out[1]:
```
```python
# Summarize the previously computed `summary` variable into a concise plain-English string.
# This cell assumes `summary` exists from prior execution (it does in the session).
metric = summary['metric']
trace = summary['trace']

# Metric summarization details
metric_row_count = metric['row_count']
metric_cmdb_ids = metric['distinct_cmdb_ids']
metric_cmdb_count = len(metric_cmdb_ids)
metric_cmdb_sample = ", ".join(metric_cmdb_ids[:8]) + (", ..." if metric_cmdb_count > 8 else "")
metric_kpi_top5 = [
    f"{row.kpi_name} ({int(row.count)})"
    for row in metric['kpi_name_counts_top'].head(5).itertuples(index=False)
]
metric_min_epoch = metric['min_timestamp_epoch']
metric_max_epoch = metric['max_timestamp_epoch']
metric_min_local = str(metric['min_timestamp_local'])
metric_max_local = str(metric['max_timestamp_local'])
metric_example_count = len(metric['example_rows'])
first_metric_example = metric['example_rows'].iloc[0].to_dict() if metric_example_count>0 else {}

# Trace summarization details
trace_row_count = trace['row_count']
trace_cmdb_ids = trace['distinct_cmdb_ids']
trace_cmdb_count = len(trace_cmdb_ids)
trace_cmdb_sample = ", ".join(trace_cmdb_ids[:8]) + (", ..." if trace_cmdb_count > 8 else "")
trace_top5 = [
    f"{row.trace_name} ({int(row.count)})"
    for row in trace['trace_name_counts_top'].head(5).itertuples(index=False)
]
trace_min_epoch = trace['min_timestamp_epoch']
trace_max_epoch = trace['max_timestamp_epoch']
trace_min_local = str(trace['min_timestamp_local'])
trace_max_local = str(trace['max_timestamp_local'])
trace_example_count = len(trace['example_rows'])
first_trace_example = trace['example_rows'].iloc[0].to_dict() if trace_example_count>0 else {}

summary_text = (
    "Summary:\n\n"
    f"metric.csv -> rows: {metric_row_count}; distinct cmdb_ids: {metric_cmdb_count} "
    f"(sample: {metric_cmdb_sample}); top 5 kpi_name (name(count)): {', '.join(metric_kpi_top5)}; "
    f"timestamp range (epoch): {metric_min_epoch} -> {metric_max_epoch}; "
    f"timestamp range (Asia/Shanghai): {metric_min_local} -> {metric_max_local}; "
    f"example rows shown: {metric_example_count} (first example: {first_metric_example}).\n\n"
    f"trace.csv  -> rows: {trace_row_count}; distinct cmdb_ids: {trace_cmdb_count} "
    f"(sample: {trace_cmdb_sample}); top 5 trace_name (name(count)): {', '.join(trace_top5)}; "
    f"timestamp range (epoch): {trace_min_epoch} -> {trace_max_epoch}; "
    f"timestamp range (Asia/Shanghai): {trace_min_local} -> {trace_max_local}; "
    f"example rows shown: {trace_example_count} (first example: {first_trace_example})."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric': {'row_count': 91127, 'distinct_cmdb_ids': ['db_001', 'db_002', 'db_003', 'db_004', 'db_005', 'db_006', 'db_007', 'db_008', 'db_009', 'db_010', 'db_011', 'db_012', 'db_013', 'docker_001', 'docker_002', 'docker_003', 'docker_004', 'docker_005', 'docker_006', 'docker_007', 'docker_008', 'os_001', 'os_002', 'os_003', 'os_004', 'os_005', 'os_006', 'os_007', 'os_008', 'os_009', 'os_010', 'os_011', 'os_012', 'os_013', 'os_014', 'os_015', 'os_016', 'os_017', 'os_018', 'os_019', 'os_020', 'os_021', 'os_022', 'osb_001', 'redis_001', 'redis_002', 'redis_003', 'redis_004', 'redis_005', 'redis_006', 'redis_007', 'redis_008', 'redis_009', 'redis_010', 'redis_011', 'redis_012'], 'kpi_name_counts_top':                     kpi_name  count
0                 Send_total   1323
1               Disk_io_util   1322
2   Incoming_network_traffic   1321
3       Processor_load_5_min   1321
4                Disk_rd_ios   1321
..                       ...    ...
45           SctRead_Per_Sec    780
46                  Row_Lock    780
47              Redo_Per_Sec    780
48        Proc_User_Used_Pct    780
49               Session_pct    780

[50 rows x 2 columns], 'min_timestamp_epoch': 1586541600, 'max_timestamp_epoch': 1586545200, 'min_timestamp_local': Timestamp('2020-04-11 02:00:00+0800', tz='Asia/Shanghai'), 'max_timestamp_local': Timestamp('2020-04-11 03:00:00+0800', tz='Asia/Shanghai'), 'example_rows':     timestamp                 local_time cmdb_id      kpi_name      value
0  1586541600  2020-04-11 02:00:00+08:00  db_001           ACS   0.020000
1  1586541600  2020-04-11 02:00:00+08:00  db_001          AIOS   0.000000
2  1586541600  2020-04-11 02:00:00+08:00  db_001           AWS   0.020000
3  1586541600  2020-04-11 02:00:00+08:00  db_001  CPU_Used_Pct   1.324514
4  1586541600  2020-04-11 02:00:00+08:00  db_001  CPU_free_pct  98.675486}, 'trace': {'row_count': 6240, 'distinct_cmdb_ids': ['docker_001', 'docker_002', 'docker_003', 'docker_004', 'docker_005', 'docker_006', 'docker_007', 'docker_008', 'os_021', 'os_022'], 'trace_name_counts_top':                                            trace_name  count
0                           trace.self.jdbc.row_count    240
1                          trace.self.jdbc.error_rate    240
2                        trace.self.jdbc.duration_p95    240
3                       trace.self.jdbc.duration_mean    240
4                          trace.self.local.row_count    240
..                                                ...    ...
45   trace.from_docker_004.remoteprocess.duration_p95     60
46     trace.from_docker_004.remoteprocess.error_rate     60
47      trace.from_docker_004.remoteprocess.row_count     60
48  trace.from_docker_001.remoteprocess.duration_mean     60
49   trace.from_docker_001.remoteprocess.duration_p95     60

[50 rows x 2 columns], 'min_timestamp_epoch': 1586543400, 'max_timestamp_epoch': 1586545140, 'min_timestamp_local': Timestamp('2020-04-11 02:30:00+0800', tz='Asia/Shanghai'), 'max_timestamp_local': Timestamp('2020-04-11 02:59:00+0800', tz='Asia/Shanghai'), 'example_rows':     timestamp                 local_time     cmdb_id                                     trace_name       value
0  1586543400  2020-04-11 02:30:00+08:00  docker_001  trace.from_os_022.remoteprocess.duration_mean    0.487008
1  1586543400  2020-04-11 02:30:00+08:00  docker_001   trace.from_os_022.remoteprocess.duration_p95    1.501850
2  1586543400  2020-04-11 02:30:00+08:00  docker_001     trace.from_os_022.remoteprocess.error_rate    0.000000
3  1586543400  2020-04-11 02:30:00+08:00  docker_001      trace.from_os_022.remoteprocess.row_count  124.000000
4  1586543400  2020-04-11 02:30:00+08:00  docker_001                   trace.self.csf.duration_mean    0.050569}}

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_metric if present, otherwise read
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure correct dtypes
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Incident window (inclusive)
incident_start = 1586543400
incident_end = 1586545200

# 1) Compute global P95 per (cmdb_id, kpi_name) using the FULL series BEFORE any filtering
global_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index()
global_p95 = global_p95.rename(columns={'value': 'global_p95'})

# Count total unique groups
total_groups = len(global_p95)

# Filter incident-window rows
df_incident = df_metric[(df_metric['timestamp'] >= incident_start) & (df_metric['timestamp'] <= incident_end)].copy()

# Merge global_p95 into incident rows to allow anomaly detection (value >= global_p95)
df_incident = df_incident.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# total_points_incident and max_value_incident from incident rows (even if no anomalies)
incident_agg = df_incident.groupby(['cmdb_id', 'kpi_name']).agg(
    total_points_incident=('value', 'size'),
    max_value_incident=('value', 'max')
).reset_index()

# anomaly rows where value >= global_p95
df_anom = df_incident[df_incident['value'] >= df_incident['global_p95']].copy()

# anomaly_count_incident and earliest_anomaly_ts_incident from anomaly rows
if not df_anom.empty:
    anom_agg = df_anom.groupby(['cmdb_id', 'kpi_name']).agg(
        anomaly_count_incident=('value', 'size'),
        earliest_anomaly_ts_incident=('timestamp', 'min')
    ).reset_index()
else:
    # empty structure
    anom_agg = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'anomaly_count_incident', 'earliest_anomaly_ts_incident'])

# Combine global_p95 with incident aggregates and anomaly aggregates
group_stats_full = global_p95.merge(incident_agg, on=['cmdb_id', 'kpi_name'], how='left') \
                            .merge(anom_agg, on=['cmdb_id', 'kpi_name'], how='left')

# Fill NaNs for groups without incident rows
group_stats_full['total_points_incident'] = group_stats_full['total_points_incident'].fillna(0).astype(int)
group_stats_full['anomaly_count_incident'] = group_stats_full['anomaly_count_incident'].fillna(0).astype(int)
# max_value_incident remains NaN if no incident points; earliest_anomaly_ts_incident NaN if no anomalies

# Add a local-time column for earliest anomaly for readability (UTC+8), keep it compact
group_stats_full['earliest_anomaly_local'] = pd.to_datetime(group_stats_full['earliest_anomaly_ts_incident'], unit='s', utc=True).dt.tz_convert(tz)

# 2) Top 20 groups ordered by anomaly_count_incident desc, then earliest_anomaly_ts_incident asc
top20 = group_stats_full.sort_values(
    by=['anomaly_count_incident', 'earliest_anomaly_ts_incident'],
    ascending=[False, True]
).head(20)

# Select and order requested columns for display
top20_display = top20[[
    'cmdb_id',
    'kpi_name',
    'global_p95',
    'total_points_incident',
    'anomaly_count_incident',
    'earliest_anomaly_ts_incident',
    'earliest_anomaly_local',
    'max_value_incident'
]]

# 3) total number of groups processed
total_groups  # will be displayed along with top20_display and group_stats_full stored for inspection

top20_display, total_groups

```
Out[2]:
```
```python
# Build a concise plain-English summary from the previously computed results.
try:
    top = top20_display  # reuse variable from kernel
    total = total_groups
except NameError:
    raise NameError("Expected variables `top20_display` and `total_groups` in the kernel. Please re-run the aggregation cell first.")

# summary facts
max_anom = int(top['anomaly_count_incident'].max())
groups_with_max = top[top['anomaly_count_incident'] == max_anom]
groups_with_max_list = groups_with_max.apply(lambda r: f"{r['cmdb_id']}:{r['kpi_name']}", axis=1).tolist()
groups_with_max_sample = ", ".join(groups_with_max_list[:5]) + (", ..." if len(groups_with_max_list) > 5 else "")

# earliest anomaly among the top20 (local time)
earliest_local = top['earliest_anomaly_local'].min()
earliest_local_str = str(earliest_local) if pd.notnull(earliest_local) else "N/A"

# how many of top20 have at least one anomaly
top20_with_anom = int((top['anomaly_count_incident'] > 0).sum())

summary_text = (
    f"Summary of metric anomaly grouping:\n\n"
    f"- Total groups processed (unique cmdb_id,kpi_name): {total}\n"
    f"- Top groups examined: {len(top)} (showing top 20 by anomaly_count_incident)\n"
    f"- Maximum anomaly_count_incident in the top20: {max_anom}\n"
    f"- Example group(s) with that max count: {groups_with_max_sample}\n"
    f"- Earliest anomaly timestamp among the top20 (Asia/Shanghai): {earliest_local_str}\n"
    f"- Number of top20 groups with at least one anomaly in the incident window: {top20_with_anom}\n\n"
    f"Notes: The incident window was timestamps >=1586543400 and <=1586545200. "
    f"global_p95 was computed using the full series for each (cmdb_id,kpi_name) before filtering."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id         kpi_name  global_p95  total_points_incident  anomaly_count_incident  earliest_anomaly_ts_incident    earliest_anomaly_local  max_value_incident
974      os_007      Disk_rd_ios    0.000000                     31                      31                  1.586543e+09 2020-04-11 02:30:00+08:00            0.000000
1636     os_020     Disk_io_util    0.000000                     31                      31                  1.586543e+09 2020-04-11 02:30:00+08:00            0.000000
2016  redis_011       redis_ping    1.000000                     31                      31                  1.586543e+09 2020-04-11 02:30:00+08:00            1.000000
10       db_001             Hang    1.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            1.000000
15       db_001        MEM_Total  381.902264                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00          381.902264
21       db_001     On_Off_State    1.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            1.000000
28       db_001         Row_Lock    0.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.000000
29       db_001  SctRead_Per_Sec    0.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.000000
46       db_002             AIOS    0.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.000000
55       db_002             Hang    1.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            1.000000
60       db_002        MEM_Total  381.902264                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00          381.902264
66       db_002     On_Off_State    1.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            1.000000
73       db_002         Row_Lock    0.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.000000
74       db_002  SctRead_Per_Sec    0.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.000000
100      db_003             Hang    1.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            1.000000
105      db_003        MEM_Total  381.902264                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00          381.902264
111      db_003     On_Off_State    1.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            1.000000
118      db_003         Row_Lock    0.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.000000
123      db_003   Sess_Used_Temp    0.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.000000
124      db_003   Sess_Used_Undo    0.000000                     30                      30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.015625, 2047)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables if present; otherwise compute baseline variables
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = df_metric['timestamp'].astype(int)
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

incident_start = 1586543400
incident_end = 1586545200

# Ensure global_p95 exists (computed from full series before filtering)
try:
    global_p95
except NameError:
    global_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value': 'global_p95'})

# Build incident-window dataframe and merge global_p95
df_incident = df_metric[(df_metric['timestamp'] >= incident_start) & (df_metric['timestamp'] <= incident_end)].copy()
df_incident = df_incident.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomaly rows: value >= global_p95
df_incident['is_anomaly'] = (df_incident['value'] >= df_incident['global_p95'])

# Aggregate some basic incident stats per group (reuse if available)
incident_agg = df_incident.groupby(['cmdb_id', 'kpi_name']).agg(
    total_points_incident=('value', 'size'),
    anomaly_count_incident=('is_anomaly', 'sum'),
    max_value_incident=('value', 'max'),
    earliest_anomaly_ts_incident=('timestamp', lambda s: s[df_incident.loc[s.index, 'is_anomaly']].min() if (df_incident.loc[s.index, 'is_anomaly'].any()) else np.nan)
).reset_index()

# earliest_anomaly_local for readability (Asia/Shanghai)
incident_agg['earliest_anomaly_local'] = pd.to_datetime(incident_agg['earliest_anomaly_ts_incident'], unit='s', utc=True).dt.tz_convert(tz)

# Compute max_consecutive_anomaly_run per group
def max_consecutive_run(subdf):
    # consider only incident-window anomaly rows for this group
    anom_ts = subdf.loc[subdf['is_anomaly'], 'timestamp'].sort_values().drop_duplicates()
    if anom_ts.empty:
        return 0
    # differences between consecutive anomaly timestamps
    diffs = anom_ts.diff().fillna(999999)
    # break when diff != 60 (one minute)
    seq_id = (diffs != 60).cumsum()
    run_lengths = seq_id.groupby(seq_id).size()
    return int(run_lengths.max())

runs = df_incident.groupby(['cmdb_id', 'kpi_name']).apply(max_consecutive_run).reset_index()
runs.columns = ['cmdb_id', 'kpi_name', 'max_consecutive_anomaly_run']

# Combine global_p95 with incident_agg and runs
grouped = global_p95.merge(incident_agg, on=['cmdb_id', 'kpi_name'], how='left').merge(runs, on=['cmdb_id', 'kpi_name'], how='left')

# Fill NaNs for groups without incident rows
grouped['total_points_incident'] = grouped['total_points_incident'].fillna(0).astype(int)
grouped['anomaly_count_incident'] = grouped['anomaly_count_incident'].fillna(0).astype(int)
grouped['max_consecutive_anomaly_run'] = grouped['max_consecutive_anomaly_run'].fillna(0).astype(int)

# Compute severity_ratio per specification
# severity_ratio = (max_value_incident - global_p95) / max(abs(max_value_incident), 1e-9)
# If max_value_incident <= global_p95, treat severity_ratio as 0
grouped['severity_ratio'] = (grouped['max_value_incident'] - grouped['global_p95']) / grouped['max_value_incident'].abs().replace(0, 1e-9)
grouped.loc[grouped['max_value_incident'].isna(), 'severity_ratio'] = 0.0
grouped.loc[grouped['max_value_incident'] <= grouped['global_p95'], 'severity_ratio'] = 0.0

# Filter to groups that had anomaly_count_incident > 0
full_anom_table = grouped[grouped['anomaly_count_incident'] > 0].copy()

# Order and select columns requested
full_anom_table = full_anom_table[[
    'cmdb_id',
    'kpi_name',
    'global_p95',
    'total_points_incident',
    'anomaly_count_incident',
    'max_consecutive_anomaly_run',
    'earliest_anomaly_ts_incident',
    'earliest_anomaly_local',
    'max_value_incident',
    'severity_ratio'
]]

# Sort by severity_ratio desc, then max_consecutive_anomaly_run desc, then anomaly_count_incident desc
top20_by_severity = full_anom_table.sort_values(
    by=['severity_ratio', 'max_consecutive_anomaly_run', 'anomaly_count_incident'],
    ascending=[False, False, False]
).head(20)

# Count of groups with anomaly_count_incident > 0
filtered_group_count = full_anom_table.shape[0]

# Display compact results: top20 table and the filtered count
top20_by_severity, filtered_group_count

```
Out[3]:
```
```python
# Summarize the top20_by_severity and filtered_group_count into a concise plain-English text.
try:
    top20 = top20_by_severity  # reuse from kernel
    total_with_anomalies = filtered_group_count
except NameError:
    raise NameError("Expected variables `top20_by_severity` and `filtered_group_count` in the kernel. Please re-run the analysis cells first.")

# Compute succinct highlights
max_sev = float(top20['severity_ratio'].max())
groups_max_sev = top20.loc[top20['severity_ratio'] == max_sev, ['cmdb_id', 'kpi_name']].apply(lambda r: f"{r['cmdb_id']}:{r['kpi_name']}", axis=1).tolist()
groups_max_sev_sample = ", ".join(groups_max_sev[:5]) + (", ..." if len(groups_max_sev) > 5 else "")

max_run = int(top20['max_consecutive_anomaly_run'].max())
groups_max_run = top20.loc[top20['max_consecutive_anomaly_run'] == max_run, ['cmdb_id', 'kpi_name']].apply(lambda r: f"{r['cmdb_id']}:{r['kpi_name']}", axis=1).tolist()
groups_max_run_sample = ", ".join(groups_max_run[:5]) + (", ..." if len(groups_max_run) > 5 else "")

earliest_local = top20['earliest_anomaly_local'].min()
earliest_local_str = str(earliest_local) if pd.notnull(earliest_local) else "N/A"

count_top20 = len(top20)
count_sev1 = int((top20['severity_ratio'] >= 0.9999).sum())  # near-1 severity

summary_text = (
    f"Anomaly summary (incident window 1586543400–1586545200):\n\n"
    f"- Total groups with at least one anomaly: {total_with_anomalies}\n"
    f"- Returning top {count_top20} groups ordered by severity_ratio (shown above).\n"
    f"- Highest severity_ratio in the top20: {max_sev:.6f} (example groups: {groups_max_sev_sample})\n"
    f"- Longest consecutive anomaly run in the top20: {max_run} points (example groups: {groups_max_run_sample})\n"
    f"- Earliest anomaly among the top20 (Asia/Shanghai): {earliest_local_str}\n"
    f"- Number of top20 groups with severity_ratio ≈ 1.0: {count_sev1}\n\n"
    f"Interpretation notes: severity_ratio = (max_value_incident - global_p95) / |max_value_incident| (treated as 0 when max <= global_p95). "
    f"Many top groups show severity_ratio near 1 and very long consecutive runs (e.g., 30), indicating sustained values above their historical 95th-percentile during the incident window."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id               kpi_name  global_p95  total_points_incident  anomaly_count_incident  max_consecutive_anomaly_run  earliest_anomaly_ts_incident    earliest_anomaly_local  max_value_incident  severity_ratio
124       db_003         Sess_Used_Undo    0.000000                     30                      30                           30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.015625        1.000000
648   docker_008     container_cpu_used    0.000000                     30                      30                           30                  1.586543e+09 2020-04-11 02:30:00+08:00            2.000000        1.000000
717       os_002             Disk_await    0.000000                     30                      30                           30                  1.586543e+09 2020-04-11 02:30:00+08:00            5.330000        1.000000
718       os_002           Disk_io_util    0.000000                     30                      30                           30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.005563        1.000000
820       os_004           Disk_io_util    0.000000                     30                      30                           30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.750000        1.000000
1074      os_009             Disk_await    0.000000                     30                      30                           30                  1.586543e+09 2020-04-11 02:30:00+08:00            2.660000        1.000000
1078      os_009             Disk_svctm    0.000000                     30                      30                           30                  1.586543e+09 2020-04-11 02:30:00+08:00            1.600000        1.000000
1535      os_018            Disk_rd_ios    0.000000                     30                      30                           30                  1.586543e+09 2020-04-11 02:30:00+08:00            6.000000        1.000000
1587      os_019            Disk_rd_kbs    0.000000                     30                      30                           30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.000394        1.000000
1607      os_019                Page_pi    0.000000                     30                      30                           30                  1.586543e+09 2020-04-11 02:30:00+08:00            0.397637        1.000000
529       db_012         Sess_Used_Undo    0.000000                     15                      15                            1                  1.586543e+09 2020-04-11 02:31:00+08:00            0.007812        1.000000
1556      os_018                Page_pi    0.082347                     30                       3                            1                  1.586543e+09 2020-04-11 02:31:00+08:00          358.207991        0.999770
1536      os_018            Disk_rd_kbs    0.000081                     30                       3                            1                  1.586544e+09 2020-04-11 02:32:00+08:00            0.349311        0.999768
621   docker_005     container_cpu_used    0.100000                     30                       3                            2                  1.586544e+09 2020-04-11 02:34:00+08:00           14.000000        0.992857
461       db_011    LFParaWrite_Per_Sec    0.160500                     30                       3                            1                  1.586543e+09 2020-04-11 02:31:00+08:00           10.730000        0.985042
58        db_002     Logic_Read_Per_Sec  299.340000                     30                       2                            1                  1.586544e+09 2020-04-11 02:42:00+08:00        16749.300000        0.982128
69        db_002  Physical_Read_Per_Sec    0.315000                     30                       2                            1                  1.586544e+09 2020-04-11 02:42:00+08:00           16.900000        0.981361
721       os_002             Disk_svctm    1.050000                     30                       1                            1                  1.586544e+09 2020-04-11 02:40:00+08:00           40.000000        0.973750
462       db_011         LFSync_Per_Sec    0.471500                     30                       3                            1                  1.586543e+09 2020-04-11 02:31:00+08:00           16.820000        0.971968
813       os_004        CPU_iowait_time    0.054398                     30                       2                            1                  1.586544e+09 2020-04-11 02:38:00+08:00            1.534735        0.964555, 1691)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing full_anom_table if present; otherwise attempt to rebuild from prior variables
try:
    full_anom_table
except NameError:
    # Try to reconstruct from grouped (which contains all groups) and filter anomaly_count_incident>0
    try:
        grouped
    except NameError:
        # Recompute necessary data from metric.csv
        df_metric = pd.read_csv('metric.csv')
        df_metric['timestamp'] = df_metric['timestamp'].astype(int)
        df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
        incident_start = 1586543400
        incident_end = 1586545200
        # global_p95 computed on full series
        global_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value': 'global_p95'})
        df_incident = df_metric[(df_metric['timestamp'] >= incident_start) & (df_metric['timestamp'] <= incident_end)].copy()
        df_incident = df_incident.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
        df_incident['is_anomaly'] = (df_incident['value'] >= df_incident['global_p95'])
        incident_agg = df_incident.groupby(['cmdb_id', 'kpi_name']).agg(
            total_points_incident=('value', 'size'),
            anomaly_count_incident=('is_anomaly', 'sum'),
            max_value_incident=('value', 'max'),
            earliest_anomaly_ts_incident=('timestamp', lambda s: s[df_incident.loc[s.index, 'is_anomaly']].min() if (df_incident.loc[s.index, 'is_anomaly'].any()) else np.nan)
        ).reset_index()
        incident_agg['earliest_anomaly_local'] = pd.to_datetime(incident_agg['earliest_anomaly_ts_incident'], unit='s', utc=True).dt.tz_convert(tz)
        def max_consecutive_run(subdf):
            anom_ts = subdf.loc[subdf['is_anomaly'], 'timestamp'].sort_values().drop_duplicates()
            if anom_ts.empty:
                return 0
            diffs = anom_ts.diff().fillna(999999)
            seq_id = (diffs != 60).cumsum()
            run_lengths = seq_id.groupby(seq_id).size()
            return int(run_lengths.max())
        runs = df_incident.groupby(['cmdb_id', 'kpi_name']).apply(max_consecutive_run).reset_index()
        runs.columns = ['cmdb_id', 'kpi_name', 'max_consecutive_anomaly_run']
        grouped = global_p95.merge(incident_agg, on=['cmdb_id', 'kpi_name'], how='left').merge(runs, on=['cmdb_id', 'kpi_name'], how='left')
        grouped['total_points_incident'] = grouped['total_points_incident'].fillna(0).astype(int)
        grouped['anomaly_count_incident'] = grouped['anomaly_count_incident'].fillna(0).astype(int)
        grouped['max_consecutive_anomaly_run'] = grouped['max_consecutive_anomaly_run'].fillna(0).astype(int)
        grouped['severity_ratio'] = (grouped['max_value_incident'] - grouped['global_p95']) / grouped['max_value_incident'].abs().replace(0, 1e-9)
        grouped.loc[grouped['max_value_incident'].isna(), 'severity_ratio'] = 0.0
        grouped.loc[grouped['max_value_incident'] <= grouped['global_p95'], 'severity_ratio'] = 0.0
    # Filter groups with anomalies
    full_anom_table = grouped[grouped['anomaly_count_incident'] > 0].copy()

# Now annotate full_anom_table with requested fields
annotated_anom_table = full_anom_table.copy()

# 1) breach_abs and breach_over_p95
annotated_anom_table['breach_abs'] = annotated_anom_table['max_value_incident'] - annotated_anom_table['global_p95']
# avoid division by zero: denom = max(global_p95, 1e-9)
annotated_anom_table['breach_over_p95'] = annotated_anom_table['breach_abs'] / annotated_anom_table['global_p95'].abs().clip(lower=1e-9)
# 2) cmdb_level extraction
prefix = annotated_anom_table['cmdb_id'].astype(str).str.split('_', n=1).str[0]
annotated_anom_table['cmdb_level'] = prefix.where(prefix.isin(['os', 'docker', 'db', 'redis']), other='other')

# Keep earliest_anomaly_local as-is (should exist); ensure dtype
if 'earliest_anomaly_local' not in annotated_anom_table.columns:
    annotated_anom_table['earliest_anomaly_local'] = pd.to_datetime(annotated_anom_table['earliest_anomaly_ts_incident'], unit='s', utc=True).dt.tz_convert(tz)

# Prepare output A: top 20 by breach_over_p95 desc, then max_consecutive_anomaly_run desc
output_A = annotated_anom_table.sort_values(
    by=['breach_over_p95', 'max_consecutive_anomaly_run'],
    ascending=[False, False]
)[[
    'cmdb_id',
    'cmdb_level',
    'kpi_name',
    'global_p95',
    'max_value_incident',
    'breach_abs',
    'breach_over_p95',
    'anomaly_count_incident',
    'max_consecutive_anomaly_run',
    'earliest_anomaly_local'
]].head(20).copy()

# Round numeric columns for compact display
num_cols = ['global_p95', 'max_value_incident', 'breach_abs', 'breach_over_p95']
for c in num_cols:
    if c in output_A.columns:
        output_A[c] = output_A[c].map(lambda v: round(float(v), 6) if pd.notnull(v) else v)

# Prepare output B: aggregation by cmdb_level
agg_B = annotated_anom_table.groupby('cmdb_level').agg(
    number_of_groups_with_anomaly=('cmdb_id', 'size'),
    total_anomaly_points_sum=('anomaly_count_incident', 'sum'),
    max_breach_over_p95=('breach_over_p95', 'max'),
    avg_breach_over_p95=('breach_over_p95', 'mean')
).reset_index()

# Round agg numeric fields for compactness
agg_B['max_breach_over_p95'] = agg_B['max_breach_over_p95'].map(lambda v: round(float(v), 6) if pd.notnull(v) else v)
agg_B['avg_breach_over_p95'] = agg_B['avg_breach_over_p95'].map(lambda v: round(float(v), 6) if pd.notnull(v) else v)

# Keep the full annotated table in kernel variable `annotated_anom_table` (already set)
# Display compact results: output_A (top 20), agg_B (by cmdb_level), and shape of full annotated table
output_A, agg_B, annotated_anom_table.shape

```
Out[4]:
```
```python
# Summarize the annotated anomaly table and aggregations in plain English.
try:
    output_A
    agg_B
    annotated_anom_table
except NameError:
    raise NameError("Expected variables `output_A`, `agg_B`, and `annotated_anom_table` in kernel. Re-run previous cells if missing.")

# Basic totals
total_groups_with_anomaly = int(annotated_anom_table.shape[0])

# Top 3 groups from output_A for examples
top_examples = output_A.head(3).copy()
top_examples_rows = []
for _, r in top_examples.iterrows():
    top_examples_rows.append(
        f"{r['cmdb_id']} | {r['kpi_name']} | global_p95={r['global_p95']} | max_value={r['max_value_incident']} | "
        f"breach_abs={r['breach_abs']} | breach_over_p95={r['breach_over_p95']} | run={int(r['max_consecutive_anomaly_run'])}"
    )
top_examples_text = "\n  - ".join(top_examples_rows)

# Aggregation summary (format rows)
agg_lines = []
for _, r in agg_B.iterrows():
    agg_lines.append(
        f"{r['cmdb_level']}: groups={int(r['number_of_groups_with_anomaly'])}, "
        f"total_anomaly_points={int(r['total_anomaly_points_sum'])}, "
        f"max_breach_over_p95={r['max_breach_over_p95']}, avg_breach_over_p95={r['avg_breach_over_p95']}"
    )
agg_text = "\n  - ".join(agg_lines)

# Note about zero/near-zero p95 producing huge ratios
note = (
    "Note: Many top 'breach_over_p95' values are extremely large because global_p95 == 0 (or very near zero) "
    "for those groups, so dividing by p95 inflates the ratio. Treat such large ratios as artifacts of a zero baseline "
    "and inspect the raw values (global_p95 and max_value) when prioritizing remediation."
)

# Check how many top groups have a full-window consecutive run (e.g., run == 30)
full_window_run_count = int((annotated_anom_table['max_consecutive_anomaly_run'] >= 30).sum())

summary_text = (
    "Anomaly breach summary (incident window 1586543400–1586545200):\n\n"
    f"- Total groups with at least one anomaly: {total_groups_with_anomaly}\n\n"
    f"- Top example groups (by breach_over_p95):\n  - {top_examples_text}\n\n"
    f"- Many top groups show very large breach_over_p95 due to global_p95 == 0; these dominate the ranking.\n"
    f"- Sustained anomalies: {full_window_run_count} groups have max_consecutive_anomaly_run >= 30 (continuous throughout much of the incident window).\n\n"
    f"- Aggregation by cmdb_level:\n  - {agg_text}\n\n"
    f"{note}\n\n"
    "Recommendation: Review groups with high raw breaches (max_value_incident and breach_abs) and long consecutive runs first, "
    "and treat extremely large breach_over_p95 values with caution if global_p95 is zero."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id cmdb_level               kpi_name  global_p95  max_value_incident    breach_abs  breach_over_p95  anomaly_count_incident  max_consecutive_anomaly_run    earliest_anomaly_local
1535      os_018         os            Disk_rd_ios    0.000000            6.000000      6.000000     6.000000e+09                      30                           30 2020-04-11 02:30:00+08:00
717       os_002         os             Disk_await    0.000000            5.330000      5.330000     5.330000e+09                      30                           30 2020-04-11 02:30:00+08:00
1074      os_009         os             Disk_await    0.000000            2.660000      2.660000     2.660000e+09                      30                           30 2020-04-11 02:30:00+08:00
648   docker_008     docker     container_cpu_used    0.000000            2.000000      2.000000     2.000000e+09                      30                           30 2020-04-11 02:30:00+08:00
1078      os_009         os             Disk_svctm    0.000000            1.600000      1.600000     1.600000e+09                      30                           30 2020-04-11 02:30:00+08:00
820       os_004         os           Disk_io_util    0.000000            0.750000      0.750000     7.500000e+08                      30                           30 2020-04-11 02:30:00+08:00
1607      os_019         os                Page_pi    0.000000            0.397637      0.397637     3.976370e+08                      30                           30 2020-04-11 02:30:00+08:00
124       db_003         db         Sess_Used_Undo    0.000000            0.015625      0.015625     1.562500e+07                      30                           30 2020-04-11 02:30:00+08:00
529       db_012         db         Sess_Used_Undo    0.000000            0.007812      0.007812     7.812000e+06                      15                            1 2020-04-11 02:31:00+08:00
718       os_002         os           Disk_io_util    0.000000            0.005563      0.005563     5.563000e+06                      30                           30 2020-04-11 02:30:00+08:00
1587      os_019         os            Disk_rd_kbs    0.000000            0.000394      0.000394     3.940000e+05                      30                           30 2020-04-11 02:30:00+08:00
1556      os_018         os                Page_pi    0.082347          358.207991    358.125644     4.348993e+03                       3                            1 2020-04-11 02:31:00+08:00
1536      os_018         os            Disk_rd_kbs    0.000081            0.349311      0.349230     4.303510e+03                       3                            1 2020-04-11 02:32:00+08:00
621   docker_005     docker     container_cpu_used    0.100000           14.000000     13.900000     1.390000e+02                       3                            2 2020-04-11 02:34:00+08:00
461       db_011         db    LFParaWrite_Per_Sec    0.160500           10.730000     10.569500     6.585358e+01                       3                            1 2020-04-11 02:31:00+08:00
58        db_002         db     Logic_Read_Per_Sec  299.340000        16749.300000  16449.960000     5.495410e+01                       2                            1 2020-04-11 02:42:00+08:00
69        db_002         db  Physical_Read_Per_Sec    0.315000           16.900000     16.585000     5.265079e+01                       2                            1 2020-04-11 02:42:00+08:00
721       os_002         os             Disk_svctm    1.050000           40.000000     38.950000     3.709524e+01                       1                            1 2020-04-11 02:40:00+08:00
462       db_011         db         LFSync_Per_Sec    0.471500           16.820000     16.348500     3.467338e+01                       3                            1 2020-04-11 02:31:00+08:00
813       os_004         os        CPU_iowait_time    0.054398            1.534735      1.480337     2.721292e+01                       2                            1 2020-04-11 02:38:00+08:00,   cmdb_level  number_of_groups_with_anomaly  total_anomaly_points_sum  max_breach_over_p95  avg_breach_over_p95
0         db                            514                      4821         1.562500e+07         4.559796e+04
1     docker                             72                      1919         2.000000e+09         2.777778e+07
2         os                            858                      6944         6.000000e+09         1.951469e+07
3      other                              2                        33         1.297668e+00         6.488340e-01
4      redis                            245                      4620         8.888890e-01         4.488400e-02, (1691, 13))```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse annotated_anom_table from kernel
try:
    annotated_anom_table
except NameError:
    raise NameError("annotated_anom_table not found in kernel. Please run prior cells to produce it.")

# Ensure earliest_anomaly_local is datetime with Asia/Shanghai tz
if 'earliest_anomaly_local' in annotated_anom_table.columns:
    annotated_anom_table['earliest_anomaly_local'] = pd.to_datetime(annotated_anom_table['earliest_anomaly_local']).dt.tz_convert(tz)
else:
    annotated_anom_table['earliest_anomaly_local'] = pd.to_datetime(annotated_anom_table['earliest_anomaly_ts_incident'], unit='s', utc=True).dt.tz_convert(tz)

# Aggregate by cmdb_id
component_agg = annotated_anom_table.groupby('cmdb_id').agg(
    cmdb_level=('cmdb_level', 'first'),
    number_of_kpis_with_anomaly=('kpi_name', 'nunique'),
    total_anomaly_points_sum=('anomaly_count_incident', 'sum'),
    max_breach_abs=('breach_abs', 'max'),
    avg_breach_abs=('breach_abs', 'mean'),
    max_consecutive_run_overall=('max_consecutive_anomaly_run', 'max'),
    earliest_anomaly_local=('earliest_anomaly_local', 'min')
).reset_index()

# Round numeric columns for compact display
for c in ['max_breach_abs', 'avg_breach_abs']:
    component_agg[c] = component_agg[c].map(lambda v: round(float(v), 6) if pd.notnull(v) else v)

# Sort by number_of_kpis_with_anomaly desc, then total_anomaly_points_sum desc
component_agg_sorted = component_agg.sort_values(
    by=['number_of_kpis_with_anomaly', 'total_anomaly_points_sum'],
    ascending=[False, False]
).reset_index(drop=True)

# Top 20
component_agg_top20 = component_agg_sorted.head(20)[[
    'cmdb_id',
    'cmdb_level',
    'number_of_kpis_with_anomaly',
    'total_anomaly_points_sum',
    'max_breach_abs',
    'avg_breach_abs',
    'max_consecutive_run_overall',
    'earliest_anomaly_local'
]]

# Total number of components aggregated
total_components = component_agg.shape[0]

# Display compact results (store full aggregated table in `component_agg` for later inspection)
component_agg_top20, total_components

```
Out[5]:
```
```python
# Build a concise plain-English summary from the component aggregation already in the kernel.
try:
    component_agg_top20
    total_components
except NameError:
    raise NameError("Expected variables `component_agg_top20` and `total_components` in the kernel. Re-run previous aggregation cell if missing.")

# Prepare top-3 highlights from the top-20 table
top3 = component_agg_top20.head(3)
top3_lines = []
for _, r in top3.iterrows():
    top3_lines.append(
        f"{r['cmdb_id']} ({r['cmdb_level']}): kpis_with_anomaly={int(r['number_of_kpis_with_anomaly'])}, "
        f"total_anomaly_points={int(r['total_anomaly_points_sum'])}, max_breach_abs={r['max_breach_abs']}, "
        f"avg_breach_abs={r['avg_breach_abs']}, max_run={int(r['max_consecutive_run_overall'])}, "
        f"earliest={str(r['earliest_anomaly_local'])}"
    )

summary = (
    f"Component-level aggregation summary:\n\n"
    f"- Total components aggregated: {int(total_components)}\n\n"
    f"- Top 20 components (ordered by number_of_kpis_with_anomaly desc, then total_anomaly_points_sum desc) are shown in the stored table.\n\n"
    f"- Top 3 examples from that table:\n  - " + "\n  - ".join(top3_lines) + "\n\n"
    f"- Observations:\n"
    f"  * Many components (both OS and DB nodes) have large numbers of affected KPIs (30–49+) and sustained anomaly runs (max_run often = 30),\n"
    f"    indicating widespread and continuous anomalies during the incident window.\n"
    f"  * Earliest anomalies for most top components occurred at 2020-04-11 02:30:00+08:00.\n\n"
    f"Use the stored full aggregated table (`component_agg`) for drill-down by specific cmdb_id and to feed the next trace-level analysis."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id cmdb_level  number_of_kpis_with_anomaly  total_anomaly_points_sum  max_breach_abs  avg_breach_abs  max_consecutive_run_overall    earliest_anomaly_local
0   os_008         os                           49                       441          2.0000        0.098818                           30 2020-04-11 02:30:00+08:00
1   os_014         os                           47                       302          2.2980        0.119889                           30 2020-04-11 02:30:00+08:00
2   os_006         os                           46                       447          2.0000        0.046589                           30 2020-04-11 02:30:00+08:00
3   os_005         os                           46                       262         87.5000        2.397110                           30 2020-04-11 02:30:00+08:00
4   os_011         os                           45                       320          3.9500        0.128916                           30 2020-04-11 02:30:00+08:00
5   os_004         os                           45                       295        475.5500       10.755105                           30 2020-04-11 02:30:00+08:00
6   db_006         db                           43                       435       2510.7550       63.166067                           30 2020-04-11 02:30:00+08:00
7   db_012         db                           43                       406        992.8450       23.180009                           30 2020-04-11 02:30:00+08:00
8   db_008         db                           43                       353       1464.3300       39.702344                           30 2020-04-11 02:30:00+08:00
9   db_001         db                           43                       352       5439.4900      128.026126                           30 2020-04-11 02:30:00+08:00
10  db_005         db                           43                       346       2273.4150       62.814465                           30 2020-04-11 02:30:00+08:00
11  os_003         os                           43                       245         53.3500        3.813775                           30 2020-04-11 02:30:00+08:00
12  db_013         db                           42                       399        709.5950       28.203690                           30 2020-04-11 02:30:00+08:00
13  os_007         os                           41                       390         15.9000        0.406669                           31 2020-04-11 02:30:00+08:00
14  db_004         db                           41                       357      38041.2100     1039.599814                           30 2020-04-11 02:30:00+08:00
15  os_012         os                           41                       290         20.1615        0.705121                           30 2020-04-11 02:30:00+08:00
16  os_001         os                           40                       411          6.9500        0.305004                           30 2020-04-11 02:30:00+08:00
17  db_011         db                           39                       503         16.3485        0.715153                           30 2020-04-11 02:30:00+08:00
18  os_009         os                           39                       354         15.4000        0.711423                           30 2020-04-11 02:30:00+08:00
19  db_002         db                           38                       382      16449.9600      658.533551                           30 2020-04-11 02:30:00+08:00, 56)```
```